# Pairwise metric comparison

One value per method: the median over samples. Spearman correlation between metrics across methods, then scatter plots of every metric pair.

## Notes

- Silhouette, CH score, marker F1, sphericity and DAPI/PolyT intensity all move together (ρ 0.7–1.0). That's one axis, so a summary score that averages them counts it five or six times. DAPI and PolyT are interchangeable.
- The more transcripts a method assigns, the worse its cells cluster. vpt 3D has the best marker F1 but assigns only ~25% of transcripts; Baysor Sopa and the raster controls assign 80–90%. Compare methods along this trade-off, not on either metric alone.
- NMP and MECR mostly follow cell size. Raster 5 µm has the best NMP of all methods and Visium the worst, so neither should be used without correcting for size.
- Area and elongation can't be compared between 2D and 3D methods: 3D footprints are unions over z planes. Use volume.
- VSI is ~0.81 for every method (the zeros are missing values) and doesn't separate methods.
- Baysor Sopa sits with the negative controls on silhouette and % undefined. Worth checking why.

In [1]:
cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]

from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from cellseg_benchmark import BASE_PATH
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from cellseg_benchmark._constants import method_colors, method_names, metric_names, methods_3D

plt.rcParams["figure.dpi"] = 120
metrics_dir = Path(BASE_PATH) / "metrics"
keys = ["sample", "method"]

order = ["n_cells", "volume_final", "area", "elongation", "sphericity",
         "intensities_DAPI", "intensities_PolyT", "Ovrlpy_stats_mean_integrity", "pct_assigned_qced",
         "negative_marker_purity", "MECR", "marker_f1",
         "silhouette_score", "calinski_harabasz_score", "Undefined",
         "maxrss_gb", "elapsed_h"]


loaders = {
    "general_stats/general_stats.csv":
        lambda d: d.query("cell_type_revised == 'all'").groupby(keys).mean(numeric_only=True),
    "assigned_transcripts/assigned_transcript_counts.csv":
        lambda d: d.groupby(keys)[["assigned_count_qced", "total_count"]].sum()
                   .eval("pct_assigned_qced = assigned_count_qced / total_count")[["pct_assigned_qced"]],
    "cell_type_metrics/clustering_score_adata_integrated_cell_type_revised.csv":
        lambda d: d.groupby(keys).mean(numeric_only=True),
    "cell_type_metrics/cell_type_distribution_adata_integrated_cell_type_revised.csv":
        lambda d: d.groupby(keys)[["Undefined"]].mean(),
    "marker_gene_metrics/negative_marker_purity_all.csv":
        lambda d: d.groupby(keys)[["negative_marker_purity"]].mean(),
    "marker_gene_metrics/MECR_score_all.csv":
        lambda d: d.query("gene1 == 'all'").groupby(keys)[["MECR"]].mean(),
    "marker_gene_metrics/marker_f1_score_cell_type_revised.csv":
        lambda d: d.groupby(keys + ["cell_type"]).f1_score.mean().groupby(keys).mean().rename("marker_f1").to_frame(),
    "Mem_and_time/mem_and_time.csv":
        lambda d: d.groupby(keys)[["maxrss_gb", "elapsed_h"]].mean(),
}

frames = []
for cohort in cohorts:
    for file, load in loaders.items():
        path = metrics_dir / cohort / file
        if not path.exists():
            continue
        try:
            d = pd.read_csv(path, index_col=0).query("sample != 'all'")
            frames.append(load(d).reset_index().melt(keys, var_name="metric").dropna())
        except Exception as e:
            print(f"skipped {cohort}/{file}: {e}")
df = pd.concat([f for f in frames if len(f)], ignore_index=True)

med = df.pivot_table(index="method", columns="metric", values="value", aggfunc="median")
med = med[[m for m in order if m in med] + sorted(set(med) - set(order))].rename(columns=metric_names)
med = med.loc[[m for m in method_colors if m in med.index]]
print(med.shape[0], "methods,", med.shape[1], "metrics")

ImportError: cannot import name 'metric_names' from 'cellseg_benchmark._constants' (/home/ubuntu/gitrepos/cellseg-benchmark/cellseg_benchmark/_constants.py)

In [ ]:
rho = med.corr(method="spearman")
g = sns.clustermap(rho.fillna(0), cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".1f",
                   annot_kws={"size": 7}, figsize=(11, 10), dendrogram_ratio=0.06,
                   cbar_pos=None)
g.ax_heatmap.set(xlabel="", ylabel="")
g.fig.suptitle("Spearman correlation between metrics, across methods", y=1.01);

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator

colors = pd.Series([method_colors.get(m, "0.6") for m in med.index], index=med.index)
markers = pd.Series(["^" if m.startswith(tuple(methods_3D)) else "o" for m in med.index], index=med.index)

def dots(x, y, **kw):
    for mk in ("o", "^"):
        i = markers == mk
        plt.scatter(x[i], y[i], c=colors[i], marker=mk, s=12, edgecolor="0.3", linewidths=0.3)

g = sns.PairGrid(med, corner=True, height=1.6, diag_sharey=False)
g.map_lower(dots)
g.map_diag(sns.kdeplot, color="0.3", lw=1, cut=0)

fmt = FuncFormatter(lambda v, _: f"{v / 1e3:g}k" if abs(v) >= 1e4 else f"{v:g}")
for ax in g.axes.flat:
    if ax is not None:
        for axis in (ax.xaxis, ax.yaxis):
            axis.set_major_locator(MaxNLocator(3))
            axis.set_major_formatter(fmt)
for ax in g.diag_axes:
    ax.set_yticks([])
    ax.spines[["left", "top", "right"]].set_visible(False)

g.fig.legend(handles=[Line2D([], [], ls="", marker=markers[m], color=c, label=method_names.get(m, m)) for m, c in colors.items()],
             loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=6)
g.fig.suptitle("Metric pairs, one point per method", y=1.01);

In [ ]:
d = med[["% Assigned", "Marker F1"]].dropna().sort_values("% Assigned", ascending=False)
front = d[d["Marker F1"] >= d["Marker F1"].cummax()]
not_ctrl = ~med.index.str.startswith("Negative_Control")

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, y in zip(axs, ["Marker F1", "NMP"]):
    plt.sca(ax)
    dots(med["% Assigned"], med[y])
    ax.set(xlabel="% Assigned", ylabel=y)
    ax.grid(alpha=0.25, lw=0.5)

axs[0].plot(front["% Assigned"], front["Marker F1"], color="0.4", lw=0.8, zorder=0)
for m, r in front.iterrows():
    axs[0].annotate(method_names.get(m, m), (r["% Assigned"], r["Marker F1"]), fontsize=6,
                    xytext=(4, 2), textcoords="offset points")
axs[0].set_title("Coverage vs quality (line: best F1 at each coverage)", fontsize=9)

x, y = med.loc[not_ctrl, ["% Assigned", "NMP"]].dropna().T.values
xs = np.linspace(x.min(), x.max(), 50)
axs[1].plot(xs, np.polyval(np.polyfit(x, y, 2), xs), color="0.4", lw=0.8, zorder=0)
axs[1].set_title("Coverage vs purity (quadratic fit, controls excluded)", fontsize=9)
fig.tight_layout();

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, y in zip(axs, ["NMP", "MECR"]):
    plt.sca(ax)
    dots(med["Volume (µm³)"], med[y])
    ax.set(xscale="log", xlabel="Volume (µm³)", ylabel=y)
    ax.grid(alpha=0.25, lw=0.5)
    for m in med.index[~not_ctrl]:
        ax.annotate(method_names.get(m, m), (med.loc[m, "Volume (µm³)"], med.loc[m, y]), fontsize=6,
                    xytext=(4, 2), textcoords="offset points")
fig.suptitle("Contamination metrics vs cell size (controls labelled)", fontsize=10)
fig.tight_layout();